In [1]:
def load_namespace():
    import sys
    sys.path.insert(1,f'/wsu/home/gy/gy40/gy4065/hm_jetscapeml_source')#WSU Grid
    sys.path.insert(1,'/content/drive/My Drive/Projects/110_JetscapeMl/hm_jetscapeml_source')#Colab GDrive v1
    sys.path.insert(1,'/content/drive/MyDrive/Projects/110_JetscapeMl/hm_jetscapeml_source')#Colab GDrive v2
    sys.path.insert(1,f'/mnt/g/My Drive/Projects/110_JetscapeMl/hm_jetscapeml_source')#wsl gdrive
    sys.path.insert(1,'G:\\My Drive\\Projects\\110_JetscapeMl\\hm_jetscapeml_source') #Windows GDrive
    sys.path.insert(1,'/home/arsalan/Projects/110_JetscapeML/hm_jetscapeml_source/') #office tower
    sys.path.insert(1,'/home/arsalan/wsu-grid/hm_jetscapeml_source') #WSU Grid fssh
    sys.path.insert(1,'/home/arsi/wsu-grid/hm_jetscapeml_source') #HmSrv1 WSL to WSU Grid fssh
    
load_namespace()

In [2]:
from IPython.display import display
print ("Dataset Preprocessor")
from jet_ml.config import Config
print(Config())

Dataset Preprocessor


2025-02-24 15:43:34.548367: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-24 15:43:34.646895: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-24 15:43:34.674499: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-24 15:43:34.854429: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-24 15:43:36.333904: W tensorflow/compiler/tf2

Directory /home/arsi/wsu-grid/hm_jetscapeml_source/models/default_simulation_name already exists.
Directory /home/arsi/wsu-grid/hm_jetscapeml_source/reports/default_simulation_name already exists.
Directory /home/arsi/wsu-grid/hm_jetscapeml_source/reports/figures/default_simulation_name already exists.
Project Root: /home/arsi/wsu-grid/hm_jetscapeml_source
Data Directory: /home/arsi/wsu-grid/hm_jetscapeml_source/data
Models Directory: /home/arsi/wsu-grid/hm_jetscapeml_source/models
Reports Directory: /home/arsi/wsu-grid/hm_jetscapeml_source/reports
Figures Directory: /home/arsi/wsu-grid/hm_jetscapeml_source/reports/figures
Simulation Models Directory: /home/arsi/wsu-grid/hm_jetscapeml_source/models/default_simulation_name
Simulation Reports Directory: /home/arsi/wsu-grid/hm_jetscapeml_source/reports/default_simulation_name
Simulation Figures Directory: /home/arsi/wsu-grid/hm_jetscapeml_source/reports/figures/default_simulation_name
Environment Details:
  TensorFlow Version: 2.17.0
  Ke

I0000 00:00:1740429821.946117     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740429822.159148     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740429822.159641     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [3]:
dataset_size=1000
# dataset_size=7200000
dataset_file_name=f"/jet_ml_benchmark_config_01_to_09_alpha_0.2_0.3_0.4_q0_1.5_2.0_2.5_MMAT_MLBT_size_{dataset_size}_balanced_unshuffled/"
# dataset_directory_name = Config().DATA_DIR / dataset_directory_name
# print(dataset_directory_name)

dataset_file_name = f"{Config().DATA_DIR}{dataset_file_name}"
# / dataset_file_name
print(dataset_file_name)

/home/arsi/wsu-grid/hm_jetscapeml_source/data/jet_ml_benchmark_config_01_to_09_alpha_0.2_0.3_0.4_q0_1.5_2.0_2.5_MMAT_MLBT_size_1000_balanced_unshuffled/


In [5]:
classes_name=[
 'MMAT_0.2_1',
 'MLBT_0.2_1.5',
 'MLBT_0.2_2.0',
 'MLBT_0.2_2.5',

 'MMAT_0.3_1',
 'MLBT_0.3_1.5',
 'MLBT_0.3_2.0',
 'MLBT_0.3_2.5',

 'MMAT_0.4_1',
 'MLBT_0.4_1.5'
 'MLBT_0.4_2.0',
 'MLBT_0.4_2.5',
 ]
print(classes_name)

['MMAT_0.2_1', 'MLBT_0.2_1.5', 'MLBT_0.2_2.0', 'MLBT_0.2_2.5', 'MMAT_0.3_1', 'MLBT_0.3_1.5', 'MLBT_0.3_2.0', 'MLBT_0.3_2.5', 'MMAT_0.4_1', 'MLBT_0.4_1.5MLBT_0.4_2.0', 'MLBT_0.4_2.5']


In [6]:
def get_label(file_path):
    import os
    parts = tf.strings.split(file_path, os.path.sep)
    return parts[-2]
def get_separated_label(file_path):
    label=get_label(file_path)# Extract the combined label (e.g., 'MLBT_0.2_1.5')
    separated_label = tf.strings.split(label, "_")  # Split the label on "_"
    return separated_label

In [7]:
import numpy as np
import tensorflow as tf
import io

def process_image(file_path):
    label = get_separated_label(file_path)

    def load_npy(path):
        path = path.numpy().decode('utf-8')  # Convert Tensor to string
        with open(path, "rb") as f:
            img = np.load(f)  # Load .npy file as NumPy array
        # print(f"Image Min: {img.min()}, Image Max: {img.max()}")  # Check values
        return img.astype(np.float32)  # Ensure correct data type

    img = tf.py_function(load_npy, [file_path], Tout=tf.float32)  # Use tf.py_function instead of tf.numpy_function
    img = tf.reshape(img, (32, 32))  # Ensure correct shape
    return img, label

In [21]:
from IPython.display import display
print ("Dataset Preprocessor: for 1 file")
img, label = process_image(f'{Config().DATA_DIR }/jet_ml_benchmark_config_01_to_09_alpha_0.2_0.3_0.4_q0_1.5_2.0_2.5_MMAT_MLBT_size_1000_balanced_unshuffled/MLBT_0.2_1.5/event_0000002.npy')
display(img)
display(label)

Dataset Preprocessor: for 1 file


<tf.Tensor: shape=(32, 32), dtype=float32, numpy=
array([[0.       , 0.       , 0.       , ..., 0.0866153, 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.0214717,
        0.       ]], dtype=float32)>

<tf.Tensor: shape=(3,), dtype=string, numpy=array([b'MLBT', b'0.2', b'1.5'], dtype=object)>

In [8]:
max_value=121.79151153564453

In [ ]:
# %%timeit
# Measure the time taken to iterate through the dataset

def find_dataset_max():
    # Step 1: Find the maximum value across the entire dataset
    print("Finding the maximum value across the entire train dataset...")
    max_value = 0.0
    for img, label in train_ds:
        max_value = max(max_value, tf.reduce_max(img).numpy())  # Get the max value of each image

# start_time = time.time()
# max_value=find_dataset_max()
# print(f"Max Value Found: {max_value}")
# end_time = time.time()
# print(f"Time taken: {end_time - start_time:.4f} seconds")

Max Value Found: 82.61666870117188


In [9]:
def scale_image(img, max_value):
    # Scale the image by dividing by the max value
    return img / max_value

In [10]:
def scale_images(img, label,max_value):
    img = scale_image(img, max_value)  # Scale the image
    return img, label

In [ ]:
#Reading all files from disk
import tensorflow as tf
tf_dataset=tf.data.Dataset.list_files(f"{dataset_file_name}*/*",shuffle=False)
image_count = len(tf_dataset)
print(image_count)

1000


I0000 00:00:1740430399.899811     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740430399.900290     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740430399.900585     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740430401.141306     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1740430401.141652     568 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-02-24

In [12]:
#reading all files from disk and processing them
import tensorflow as tf
tf_dataset=tf.data.Dataset.list_files(f"{dataset_file_name}*/*",shuffle=False).map(process_image).map(lambda img, label: scale_images(img, label, max_value))
image_count = len(tf_dataset)
print(image_count)

1000


In [28]:
for file in tf_dataset.take(3):
    print(file[1].numpy())

[b'MLBT' b'0.2' b'1.5']
[b'MLBT' b'0.2' b'1.5']
[b'MLBT' b'0.2' b'1.5']


In [29]:
train_size = int(0.9 * image_count)
train_ds = tf_dataset.take(train_size)
print(len(train_ds))
test_ds = tf_dataset.skip(train_size)
print(len(test_ds))

900
100


In [30]:
import time

In [31]:
print("showing 1 image and label")
for image, label in train_ds.take(1):
    print("****",image)
    print("****",label)

showing 1 image and label
**** tf.Tensor(
[[0.         0.         0.         ... 0.         0.         0.00810621]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.00933275 ... 0.         0.         0.        ]], shape=(32, 32), dtype=float32)
**** tf.Tensor([b'MLBT' b'0.2' b'1.5'], shape=(3,), dtype=string)


## Implmenting k folding and shuffling for training